In [ ]:
import os
import sys

import pandas as pd

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.oasisb_utils import (
    APM_MIME_TYPES_SIDECAR,  # only for cross-referencing between apm and em collections
    APM_MIME_TYPES_SOLITARY,
    EM_HFIVE_MIME_TYPES_SIDECAR,  # hdf
    EM_HFIVE_MIME_TYPES_SOLITARY,
    EM_IMAGE_MIME_TYPES_SIDECAR,  # image
    EM_IMAGE_MIME_TYPES_SOLITARY,
    EM_MIXED_MIME_TYPES_SIDECAR,  # mixed, spectrum, etc.
    EM_MIXED_MIME_TYPES_SOLITARY,
    EM_MTEX_MIME_TYPES_SIDECAR,  # mtex
    EM_MTEX_MIME_TYPES_SOLITARY,
    get_project_id,
    prepare_parsing,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

## Decompress the original files from the scientists from the storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [ ]:
config: dict[str, str] = {
    "python_version": f"{sys.version.replace(' ', '_')}",
    "working_directory": f"{os.getcwd()}",
    "pynxtools_em version": f"{get_pynxtools_em_version()}",
    # "directory": f"src_directory,  # sys.argv[1],
}

spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")
project_range: tuple[int, int] = (1, 871)

# keep_searching_toggle = True
cnt = 0  # how many files to decompress
vol = 0  # how much byte volume does this add to scratch
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use == "1":
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.project_name) <= project_range[1]:
            project_id = get_project_id(f"{row.project_name}")
            # print(f"project{SEPARATOR}{project_id}{SEPARATOR}decompress...")

            status = prepare_parsing(
                f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                src_directory,
                project_id,
                trg_directory,
                report=True,
                write=False,
                mime_type="apm",  # "image",  # "mtex", "hdf", "image", "mixed"
                mime_type_solitary=APM_MIME_TYPES_SOLITARY,
                mime_type_sidecar=APM_MIME_TYPES_SIDECAR,
            )
            for key, obj in status.items():
                if obj["n"] > 0:
                    print(f"{project_id}, {key}, {obj['n']}, {obj['bytes']}")
                    cnt += obj["n"]
                    vol += obj["bytes"]
print(f"{cnt} files, if uncompressed {vol / 1024**3} GiB")
print("Batch queue completed")

***